In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
class Value:
    def __init__(self, data, _childern=(), _op='', label=''):
        self.data = data
        self.grad=0.0
        self._backward= lambda : None
        self._prev = set(_childern)
        self._op= _op
        self.label= label

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad+= 1*out.grad
            other.grad+= 1*out.grad
            
        out._backward =_backward
        
        return out
    
    

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other),'*')
        
        def _backward():
            self.grad+= other.data*out.grad
            other.grad+= self.data*out.grad
            
        out._backward =_backward
        
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only whole number"
        out = Value(self.data ** other, (self,), f'**{other}')
        
        def _backward():
            self.grad += other * (self.data**(other-1)) * out.grad
        
        out._backward =_backward
        return out

    def __rmul__(self, other):
        return self*other
    
    def __radd__(self, other):
        return self + other
    
    def __truediv__(self, other):
        return self * other**-1
    
    def __neg__(self):
        return self*(-1)
    
    def __sub__(self, other):
        return self + (-other)

    def tanh(self):
        x=self.data
        t=((math.exp(2*x)-1)/(math.exp(2*x)+1))
        out= Value(t,(self, ), 'tanh')
        
        def _backward():
            self.grad += (1-t**2)*out.grad
            
        out._backward =_backward
        return out
    
    def exp(self):
        x=self.data
        out= Value(math.exp(x),(self, ), 'exp')
        
        def _backward():
            self.grad += out.data*out.grad
            
        out._backward =_backward
        return out
    

    def backward(self):
        
        topo=[]
        visited=set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad=1
        
        for node in reversed(topo):
            node._backward()
        


In [ ]:
import os

# Update this path if Graphviz is installed in a different folder
os.environ["PATH"] += os.pathsep + r"C:\Users\ankit\AppData\Local\Graphviz\bin"

import graphviz
# Your graphviz code here

In [ ]:
from graphviz import Digraph

def trace(root):
    # builds a set of all nodes and edges in a graph
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
    
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph, create a rectangular ('record') node for it
        dot.node(name = uid, label = "{%s| data %.4f| grad %.4f }" % (n.label, n.data, n.grad ), shape='record')
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name = uid + n._op, label = n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [ ]:
# input
x1 = Value(2.0, label="x1")
x2 = Value(0.0, label="x2")

# weight
w1 = Value(-3.0, label="w1")
w2 = Value(1.0, label="w2")

#bias
b = Value(6.8813735870195432, label="b")

x1w1 = x1 * w1
x1w1.label = "x1*w1"
x2w2 = x2 * w2
x2w2.label = "x2*w2"
x1w1x2w2 = x1w1 + x2w2
x1w1x2w2.label = "x1*w1+x2*w2"


n=x1w1x2w2+b; n.label='n'


#---
e= (2*n).exp()
o= (e-1)/(e+1)
#---

o.label='o'
o.backward()


In [ ]:
draw_dot(o)

In [ ]:
import random

In [213]:

class Neuron:
    def __init__(self, nin):
        self.w= [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b= Value(random.uniform(-1,1))
    
    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out= act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]
    

class Layer:
    def __init__(self, nin , nout):
        self.neurons= [Neuron(nin) for _ in range(nout)]
        
    def __call__(self,x):
        outs= [n(x) for n in self.neurons]
        return outs[0] if len(outs)==1 else outs
    
    def parameters(self):
        params=[]
        for neuron in self.neurons:
            ps= neuron.parameters()
            params.extend(ps)
        return params
    

class MLP:
    def __init__(self, nin, nouts):
        sz= [nin] + nouts
        self.layers= [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
        
    def __call__(self, x):
        for layer in self.layers:
            x= layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]



In [214]:
x= [2.0, 3.0, -1.0]
n= MLP(3, [4,4,1])
n(x)

Value(data=-0.14431210642452127)

In [215]:
xs = [
    [2.0, 3.0, -1.0], 
    [3.0, -1.0, 0.5], 
    [0.5, 1.0, 1.0], 
    [1.0, -1.0, -1.0]
]
ys = [1.0, -1.0, -1.0, 1.0]  # desired output

In [216]:
y_pred=[n(x) for x in xs]
y_pred

[Value(data=-0.14431210642452127),
 Value(data=0.4069140709249298),
 Value(data=-0.5299247807432099),
 Value(data=-0.08935268554763606)]

In [225]:
for k in range(20):
    
    # forward pass
    y_pred=[n(x) for x in xs]
    loss = sum([(yout - ygt) ** 2 for ygt, yout in zip(ys, y_pred)])
    
    # set grad =0 because we do += on grad
    
    for p in n.parameters():
        p.grad=0
    
    # backward pass
    loss.backward()
    
    #update
    for p in n.parameters():
        p.data += -0.05 * p.grad
    
    print(k,"-->" , loss.data)


0 --> 0.007734850782994362
1 --> 0.007654764618348913
2 --> 0.007576268295188766
3 --> 0.007499315626763071
4 --> 0.007423862182482639
5 --> 0.007349865205739043
6 --> 0.007277283536275079
7 --> 0.007206077536816666
8 --> 0.0071362090236962985
9 --> 0.007067641201218449
10 --> 0.007000338599533336
11 --> 0.00693426701580411
12 --> 0.006869393458465367
13 --> 0.006805686094386233
14 --> 0.0067431141987633215
15 --> 0.006681648107580899
16 --> 0.006621259172487168
17 --> 0.006561919717944285
18 --> 0.00650360300052052
19 --> 0.0064462831702010795


In [226]:
y_pred

[Value(data=0.9503005963149801),
 Value(data=-0.9474237148759543),
 Value(data=-0.9901604877660282),
 Value(data=0.9666058285758266)]